<h2 align='center'>Codebasics ML Course: ML Flow Tutorial</h2>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

#### Handle class imbalance

In [5]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

### Track Experiments

In [6]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [7]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [8]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [ ]:
# Initialize MLflow
mlflow.set_experiment("Anomaly Detection2")
mlflow.set_tracking_uri("http://localhost:5000")  # for local without database
#mlflow.set_tracking_uri("sqlite:///mlflow.db") # for local with database

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(
                model,
                "model",
                skops_trusted_types=["sklearn.tree._tree.Tree"],
            )

2026/10/19 17:13:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/19 17:13:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/3/runs/4fea11d3b10a4709a8be1fd80ca9c7ac
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/10/19 17:14:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/3/runs/8ba913349d0b46508e37e2808d6cb8dd
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/10/19 17:14:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/3/runs/e5b548159d324366a73a11fc44cd426b
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/3/runs/ea8d1be1434446d1b6220c2f7b1e70e8
🧪 View experiment at: http://localhost:5000/#/experiments/3


### Register the Model

In [15]:
model_name = 'XGB-Smote'
run_id=input('Please type RunID')
#model_uri = f'runs:/{run_id}/model_name'

#with mlflow.start_run(run_id=run_id):
#    mlflow.register_model(model_uri=model_uri, name=model_name)

model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/10/19 17:45:07 WARNING mlflow.tracking._model_registry.fluent: Run with id ea8d1be1434446d1b6220c2f7b1e70e8 has no artifacts at artifact path 'model', registering model based on models:/m-23719adf196d457eb976f7f21a8d4a2f instead
2026/10/19 17:45:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 3
Created version '3' of model 'XGB-Smote'.


<ModelVersion: aliases=[], creation_timestamp=1792403107926, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1792403107926, metrics=None, model_id=None, name='XGB-Smote', params=None, run_id='ea8d1be1434446d1b6220c2f7b1e70e8', run_link='', source='models:/m-23719adf196d457eb976f7f21a8d4a2f', status='READY', status_message=None, tags={}, user_id='', version='3', workspace='default'>

### Load the Model

In [23]:
model_name = "XGB-Smote"
#model_version = 3 #1
#model_uri = f"models:/{model_name}/{model_version}"

model_uri = f"models:/{model_name}@challenger"
loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

### Transition the Model to Production

In [24]:
current_model_uri = f"models:/{model_name}@challenger"
production_model_name = "anomaly-detection-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

Successfully registered model 'anomaly-detection-prod'.
Copied version '3' of model 'XGB-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1792416594968, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1792416594968, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='ea8d1be1434446d1b6220c2f7b1e70e8', run_link='', source='models:/XGB-Smote/3', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [25]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

Please refer to following to learn more about model registry

https://mlflow.org/docs/latest/model-registry.html#model-registry-workflows to learn 

In [26]:
mlflow.end_run()